# Hakbang PH — Career Recommendation ML Lab

This Google Colab notebook builds an explainable proof-of-concept career recommender for professionals in the Philippine context. It:

- creates exactly **100 deterministic synthetic professional profiles**;
- compares five classification algorithms with stratified 5-fold cross-validation;
- selects and trains the strongest model by mean macro-F1, then accuracy;
- produces three ranked career moves from experience, skills, industry, and source-graded current/future demand;
- explains an AI-era opportunity, the human judgment that still matters, and a portfolio proof for each move;
- links an official credential and a clearly caveated practitioner account;
- exports the data, benchmark, and fitted model.

> **Important limitation:** The profiles and their labels are synthetic. Cross-validation measures how well a model recovers the rules used to create this demo data; it does **not** establish real-world career-placement accuracy. Demand grades summarize dated public evidence; they are not live vacancy counts, forecasts, employment probabilities, or guarantees. Do not use this prototype for hiring, screening, promotion, redundancy, compensation, or another high-impact decision.

**No runtime text generation:** research claims, certification names, URLs, and practitioner accounts come from the fixed evidence registry in this notebook. If a role lacks direct Philippine evidence, the notebook labels the inference instead of presenting it as a fact.


## 1. Setup

Colab normally includes the scientific Python stack. The next cell installs a package only if the import is missing.


In [ ]:
import importlib
import importlib.util
import subprocess
import sys

REQUIRED_PACKAGES = {
    "numpy": "numpy",
    "pandas": "pandas",
    "sklearn": "scikit-learn",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
    "joblib": "joblib",
}

missing = [
    package
    for module, package in REQUIRED_PACKAGES.items()
    if importlib.util.find_spec(module) is None
]
if missing:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "--quiet", *missing]
    )

from copy import deepcopy
from datetime import date
from urllib.parse import urlparse

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
try:
    from IPython.display import Markdown, display
except ImportError:
    class Markdown(str):
        pass

    def display(value):
        print(value)
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import (
    StratifiedKFold,
    cross_val_predict,
    cross_validate,
)
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SEED = 20260725
DATASET_VERSION = "PH-COLAB-SYN-2026.07.27"
np.random.seed(SEED)
sns.set_theme(style="whitegrid")
pd.set_option("display.max_colwidth", 120)

print("Environment ready.")
print("Python:", sys.version.split()[0])
print("scikit-learn:", importlib.import_module("sklearn").__version__)


## 2. Transparent schema and evidence registry

Certification recommendations come only from a fixed registry. Each credential URL points to its issuing organization's official site and was manually checked on **27 July 2026**. Practitioner outcomes are labeled by source type and include a limitation. A production service should recheck link status, exam availability, prerequisites, pricing, and retirement dates on a schedule.

Core evidence:

- [Philippine Statistics Authority — May 2026 Labor Force Survey](https://psa.gov.ph/statistics/labor-force-survey?vcode=sl76S8)
- [Philippine Standard Occupational Classification](https://psa.gov.ph/classifications-api/psoc)
- [DOLE Bureau of Local Employment — Jobs and Labor Market Forecast](https://ble.dole.gov.ph/jobs-and-labor-market-forecast/)
- [TESDA — TVET Skills Insights: 5th Industrial Revolution](https://www.tesda.gov.ph/Uploads/File/SkillInsights/2025/TVET%20Skills%20Insights%20Report%20_%205th%20Industrial%20Revolution.pdf)
- [World Economic Forum — Future of Jobs 2025: Jobs Outlook](https://www.weforum.org/publications/the-future-of-jobs-report-2025/in-full/2-jobs-outlook/)
- [World Economic Forum — Future of Jobs 2025: Skills Outlook](https://www.weforum.org/publications/the-future-of-jobs-report-2025/in-full/3-skills-outlook/)
- [International Labour Organization — Generative AI and Jobs: A Refined Global Index](https://www.ilo.org/publications/generative-ai-and-jobs-refined-global-index-occupational-exposure)

The PSA survey supplies current national labor context, not occupation-level demand for every pathway. WEF and ILO evidence is global and directional. Those distinctions are preserved in each role's `basis` field.


In [ ]:
INDUSTRIES = [
    "it_bpo",
    "financial_services",
    "manufacturing_logistics",
    "retail_ecommerce",
    "government_public",
    "healthcare",
    "education",
    "tourism_hospitality",
    "professional_services",
    "other",
]

INDUSTRY_LABELS = {
    "it_bpo": "IT–BPM / BPO",
    "financial_services": "Financial services",
    "manufacturing_logistics": "Manufacturing & logistics",
    "retail_ecommerce": "Retail & e-commerce",
    "government_public": "Government & public sector",
    "healthcare": "Healthcare",
    "education": "Education",
    "tourism_hospitality": "Tourism & hospitality",
    "professional_services": "Professional services",
    "other": "Other / cross-industry",
}

NUMERIC_FEATURES = [
    "years_experience",
    "leadership_years",
    "data_analytics",
    "technology",
    "communication",
    "project_management",
    "creative_design",
    "finance",
    "people_hr",
    "operations",
    "customer_experience",
]
CATEGORICAL_FEATURES = ["current_industry"]
SKILL_FEATURES = NUMERIC_FEATURES[2:]

EVIDENCE_SOURCES = {
    "psa_lfs": {
        "name": "May 2026 Labor Force Survey",
        "owner": "Philippine Statistics Authority",
        "url": "https://psa.gov.ph/statistics/labor-force-survey?vcode=sl76S8",
        "published": "8 Jul 2026",
    },
    "dole_forecast": {
        "name": "Jobs and Labor Market Forecast",
        "owner": "DOLE Bureau of Local Employment",
        "url": "https://ble.dole.gov.ph/jobs-and-labor-market-forecast/",
        "published": "2023–2025 release",
    },
    "tesda_5ir": {
        "name": "TVET Skills Insights: 5th Industrial Revolution",
        "owner": "TESDA",
        "url": "https://www.tesda.gov.ph/Uploads/File/SkillInsights/2025/TVET%20Skills%20Insights%20Report%20_%205th%20Industrial%20Revolution.pdf",
        "published": "2025",
    },
    "wef_jobs": {
        "name": "Future of Jobs 2025 — Jobs Outlook",
        "owner": "World Economic Forum",
        "url": "https://www.weforum.org/publications/the-future-of-jobs-report-2025/in-full/2-jobs-outlook/",
        "published": "8 Jan 2025",
    },
    "wef_skills": {
        "name": "Future of Jobs 2025 — Skills Outlook",
        "owner": "World Economic Forum",
        "url": "https://www.weforum.org/publications/the-future-of-jobs-report-2025/in-full/3-skills-outlook/",
        "published": "8 Jan 2025",
    },
    "wef_industry": {
        "name": "Future of Jobs 2025 — Industry Insights",
        "owner": "World Economic Forum",
        "url": "https://www.weforum.org/publications/the-future-of-jobs-report-2025/in-full/5-region-economy-and-industry-insights/",
        "published": "8 Jan 2025",
    },
    "ilo_genai": {
        "name": "Generative AI and Jobs: A Refined Global Index",
        "owner": "International Labour Organization",
        "url": "https://www.ilo.org/publications/generative-ai-and-jobs-refined-global-index-occupational-exposure",
        "published": "20 May 2025",
    },
    "pmi_talent": {
        "name": "PMP salary and talent survey",
        "owner": "Project Management Institute",
        "url": "https://www.pmi.org/about/press-media/2025/pmp-certification-holders-build-career-momentum-and-experience-earning-advantage-pmi-survey-finds",
        "published": "13 Nov 2025",
    },
    "isc2_cc": {
        "name": "Who earns the ISC2 CC?",
        "owner": "ISC2",
        "url": "https://www.isc2.org/insights/2025/11/who-earns-the-isc2-certified-in-cybersecurity-certification",
        "published": "3 Nov 2025",
    },
}

def demand(label, score, basis, insight, sources):
    return {
        "label": label,
        "score": score,
        "basis": basis,
        "insight": insight,
        "sources": sources,
    }

PORTFOLIO_CAVEAT = (
    "A credential can validate knowledge; it does not replace role-relevant "
    "projects, supervised practice, or measurable work outcomes."
)

CAREERS = {
    "data_bi_analyst": {
        "title": "Data & Business Intelligence Analyst",
        "min_experience": 2,
        "preferred_industries": ["financial_services", "it_bpo", "retail_ecommerce"],
        "current_demand": demand(
            "Strong", 0.8, "Sector-to-role inference",
            "DOLE identifies IT–BPM/BPO as an in-demand Philippine sector, while TESDA names big-data work among high-growth occupations. The BI mapping is an inference, not a vacancy count.",
            ["dole_forecast", "tesda_5ir"],
        ),
        "future_demand": demand(
            "Very strong", 1.0, "Global directional evidence",
            "WEF places Big Data Specialists first among the fastest-growing roles through 2030 and AI and big data first among rising skills.",
            ["wef_jobs", "wef_skills"],
        ),
        "ai_opportunity": "Use copilots to draft measures, explain variance, and accelerate exploration, then own metric definitions, data quality, and decision context.",
        "human_edge": "Stakeholder framing and checking whether a statistically correct output answers the real business question.",
        "first_proof": "Build one decision dashboard with a documented data dictionary, AI-use log, and before/after business metric.",
        "certification": {
            "name": "Microsoft Certified: Power BI Data Analyst Associate",
            "issuer": "Microsoft",
            "url": "https://learn.microsoft.com/en-us/credentials/certifications/data-analyst-associate/",
            "eligibility": "Intermediate credential; review the current PL-300 exam page.",
            "why_it_fits": "PL-300 assesses data preparation, modeling, visualization, analysis, management, and security in Power BI.",
            "practitioner": "Sarah Krusleski · Power BI practitioner",
            "practitioner_insight": "She reported recruiter messages for roles that explicitly required PL-300 and inquiries about teaching Power BI; she also says certification is not mandatory.",
            "practitioner_source_type": "Public first-person account",
            "practitioner_url": "https://www.linkedin.com/posts/sekrusleski_what-will-passing-the-pl-300-microsoft-power-activity-7269734663479279616-quT4",
            "practitioner_caveat": PORTFOLIO_CAVEAT,
        },
    },
    "cybersecurity_analyst": {
        "title": "Cybersecurity Analyst",
        "min_experience": 2,
        "preferred_industries": ["it_bpo", "financial_services", "government_public"],
        "current_demand": demand(
            "Strong", 0.8, "Direct role evidence",
            "ISC2 reports entry-level CC holders in analyst and security-operations pathways; DOLE separately identifies IT–BPM/BPO as an in-demand Philippine sector.",
            ["isc2_cc", "dole_forecast"],
        ),
        "future_demand": demand(
            "Very strong", 1.0, "Global directional evidence",
            "WEF lists Information Security Analysts among the 15 fastest-growing roles and networks and cybersecurity as the second-fastest-rising skill group through 2030.",
            ["wef_jobs", "wef_skills"],
        ),
        "ai_opportunity": "Apply AI to alert triage, threat research, and control documentation while testing AI systems for prompt injection, data leakage, and model abuse.",
        "human_edge": "Risk judgment, incident accountability, and adversarial reasoning when automated output is incomplete or misleading.",
        "first_proof": "Create an incident-response lab and publish a redacted write-up covering detection, evidence, containment, and lessons learned.",
        "certification": {
            "name": "ISC2 Certified in Cybersecurity (CC)",
            "issuer": "ISC2",
            "url": "https://www.isc2.org/certifications/cc",
            "eligibility": "No prior work experience is required; ISC2 says a new exam outline starts 1 Sep 2026.",
            "why_it_fits": "CC validates foundational security principles for an entry or adjacent move.",
            "practitioner": "Lance Rosengarten, CC · SOC Analyst",
            "practitioner_insight": "He wrote that, weeks after passing CC, he entered a GRC internship and was later offered a SOC Analyst Team Lead role.",
            "practitioner_source_type": "Issuer-published holder account",
            "practitioner_url": "https://www.isc2.org/Insights/2024/02/My-Journey-into-Cybersecurity-With-ISC2",
            "practitioner_caveat": "This sequence does not prove CC caused the outcome. He also used labs, self-study, and an internship.",
        },
    },
    "cloud_solutions_engineer": {
        "title": "Cloud Solutions Engineer",
        "min_experience": 3,
        "preferred_industries": ["it_bpo", "financial_services", "professional_services"],
        "current_demand": demand(
            "Strong", 0.8, "Sector-to-role inference",
            "DOLE identifies IT–BPM/BPO as in demand. Cloud engineering is treated as enabling infrastructure, not counted as a live national vacancy total.",
            ["dole_forecast", "tesda_5ir"],
        ),
        "future_demand": demand(
            "Very strong", 1.0, "Sector-to-role inference",
            "WEF says information and technology services employers expect near-universal AI and information-processing adoption by 2030. Cloud-architecture demand is an inference from that adoption.",
            ["wef_industry"],
        ),
        "ai_opportunity": "Design the governed data, identity, security, observability, and cost controls that let teams run AI workloads reliably.",
        "human_edge": "Architectural trade-offs across resilience, security, latency, cost, and regulation.",
        "first_proof": "Deploy a retrieval-based AI service with least-privilege access, monitoring, a cost budget, and an architecture decision record.",
        "certification": {
            "name": "AWS Certified Solutions Architect – Associate",
            "issuer": "Amazon Web Services",
            "url": "https://aws.amazon.com/certification/certified-solutions-architect-associate/",
            "eligibility": "AWS recommends prior hands-on experience; review the official exam guide.",
            "why_it_fits": "The credential validates design of secure, resilient, high-performing, and cost-optimized AWS solutions.",
            "practitioner": "Siddharth Pasumarthy · AWS Solutions Architect",
            "practitioner_insight": "He says hands-on labs broadened his technical range; after more than a year of experience and the certification, he accepted an AWS Solutions Architect offer.",
            "practitioner_source_type": "Issuer-published holder account",
            "practitioner_url": "https://aws.amazon.com/blogs/training-and-certification/steps-to-start-your-aws-certification-journey/",
            "practitioner_caveat": "Substantial self-learning and hands-on platform experience were also part of the transition.",
        },
    },
    "project_manager": {
        "title": "Project Manager",
        "min_experience": 4,
        "preferred_industries": ["professional_services", "it_bpo", "manufacturing_logistics"],
        "current_demand": demand(
            "Strong", 0.8, "Global directional evidence",
            "PMI's 2025 research describes project talent as cross-industry and reports sustained global demand; it is not a Philippines-only vacancy measure.",
            ["pmi_talent"],
        ),
        "future_demand": demand(
            "Strong", 0.8, "Global directional evidence",
            "PMI projects that the world may need up to 30 million additional project professionals by 2035 as organizations deliver AI and other transformations.",
            ["pmi_talent"],
        ),
        "ai_opportunity": "Use AI for draft plans, status synthesis, risk prompts, and meeting follow-through while keeping humans accountable for scope, value, and escalation.",
        "human_edge": "Negotiation, change leadership, judgment under uncertainty, and ownership of outcomes.",
        "first_proof": "Lead a small AI-enabled process change with a benefits baseline, risk register, adoption plan, and post-implementation review.",
        "certification": {
            "name": "Project Management Professional (PMP)®",
            "issuer": "Project Management Institute",
            "url": "https://www.pmi.org/certifications/project-management-pmp",
            "eligibility": "Experience and training requirements apply; consider CAPM if not yet eligible.",
            "why_it_fits": "PMP tests people, process, and business-environment capabilities across predictive, agile, and hybrid delivery.",
            "practitioner": "14,628 certification holders and peers across 21 countries",
            "practitioner_insight": "PMI's 2025 survey reported a 17% higher median salary for PMP holders than non-certified respondents.",
            "practitioner_source_type": "Issuer survey",
            "practitioner_url": "https://www.pmi.org/about/press-media/2025/pmp-certification-holders-build-career-momentum-and-experience-earning-advantage-pmi-survey-finds",
            "practitioner_caveat": "The result is an association from an issuer survey, not proof that PMP caused the pay difference.",
        },
    },
    "digital_marketing_strategist": {
        "title": "Digital Marketing & E-commerce Strategist",
        "min_experience": 2,
        "preferred_industries": ["retail_ecommerce", "professional_services", "tourism_hospitality"],
        "current_demand": demand(
            "Strong", 0.8, "Sector-to-role inference",
            "DOLE identifies platform work, services, tourism, and IT–BPM/BPO among areas of Philippine demand; the marketing-role mapping is an inference.",
            ["dole_forecast"],
        ),
        "future_demand": demand(
            "Strong", 0.8, "Global directional evidence",
            "WEF identifies AI and big data, technological literacy, and creative thinking as rising skills. These support AI-enabled marketing work but do not constitute a role-level forecast.",
            ["wef_skills"],
        ),
        "ai_opportunity": "Use AI for research synthesis, content variants, experimentation, and analysis while governing brand accuracy, consent, and measurement.",
        "human_edge": "Audience empathy, positioning, editorial judgment, and accountable interpretation of causal evidence.",
        "first_proof": "Run a small campaign experiment with a hypothesis, human-reviewed AI workflow, attribution limits, and documented learning.",
        "certification": {
            "name": "Meta Certified Digital Marketing Associate",
            "issuer": "Meta Blueprint",
            "url": "https://www.facebookblueprint.com/student/path/517001-get-certified-as-digital-marketing-associate",
            "eligibility": "Entry-level certification; confirm current language and exam availability.",
            "why_it_fits": "The credential covers Meta advertising fundamentals, targeting, creative, optimization, and measurement.",
            "practitioner": "Ahmad · public first-person author",
            "practitioner_insight": "He wrote that the certificate alone produced no offers and that applying the learning in visible work mattered more.",
            "practitioner_source_type": "Public first-person account",
            "practitioner_url": "https://medium.com/write-a-catalyst/i-passed-the-meta-certification-and-waited-for-my-life-to-change-it-didnt-until-i-did-this-1992e64e7404",
            "practitioner_caveat": "The author's identity and outcome are not independently verified; the account is included as a counterexample to credential guarantees.",
        },
    },
    "people_analytics_specialist": {
        "title": "People Analytics Specialist",
        "min_experience": 3,
        "preferred_industries": ["it_bpo", "professional_services", "financial_services"],
        "current_demand": demand(
            "Moderate", 0.6, "Sector-to-role inference",
            "Philippine evidence is sector-level rather than a reviewed national occupation count. Analytics-intensive employers create a plausible adjacent pathway.",
            ["dole_forecast", "tesda_5ir"],
        ),
        "future_demand": demand(
            "Strong", 0.8, "Global directional evidence",
            "WEF identifies AI and big data, analytical thinking, and talent management among important or rising skills; the role-level conclusion is directional.",
            ["wef_skills"],
        ),
        "ai_opportunity": "Use AI to accelerate workforce analysis and employee-query synthesis while enforcing privacy, fairness checks, and human review.",
        "human_edge": "Employment context, ethical judgment, causal interpretation, and trust with workers and leaders.",
        "first_proof": "Build a privacy-preserving retention analysis with a data dictionary, bias checks, uncertainty notes, and an action recommendation.",
        "certification": {
            "name": "SHRM People Analytics Specialty Credential",
            "issuer": "Society for Human Resource Management",
            "url": "https://www.shrm.org/credentials/specialty-credentials/people-analytics-credential",
            "eligibility": "A structured program and final knowledge assessment are required.",
            "why_it_fits": "The program connects people-data analysis to HR decisions and stakeholder communication.",
            "practitioner": "Cathy Evans · SHRM learner",
            "practitioner_insight": "Her issuer-published testimonial describes greater confidence and motivation after applying the course material.",
            "practitioner_source_type": "Issuer-published holder account",
            "practitioner_url": "https://www.shrm.org/gl/shop/product.html/shrm-people-analytics-specialty-credential-p",
            "practitioner_caveat": "This is issuer-selected learning feedback, not an independently measured career outcome.",
        },
    },
    "supply_chain_analyst": {
        "title": "Supply Chain Analyst",
        "min_experience": 2,
        "preferred_industries": ["manufacturing_logistics", "retail_ecommerce", "other"],
        "current_demand": demand(
            "Strong", 0.8, "Sector-to-role inference",
            "DOLE identifies construction, engineering, services, and related operational work as in demand; supply-chain analysis is an adjacent inference rather than a live vacancy count.",
            ["dole_forecast", "psa_lfs"],
        ),
        "future_demand": demand(
            "Strong", 0.8, "Global directional evidence",
            "WEF expects AI, big data, and technological literacy to rise while operational resilience remains important across industries.",
            ["wef_skills", "wef_industry"],
        ),
        "ai_opportunity": "Use AI for demand sensing, exception triage, scenario generation, and supplier research while monitoring data drift and operational constraints.",
        "human_edge": "Trade-off decisions across service, inventory, resilience, cost, and supplier relationships.",
        "first_proof": "Build a forecast-and-inventory scenario with error metrics, stockout/cost trade-offs, and a documented human override policy.",
        "certification": {
            "name": "APICS Certified in Planning and Inventory Management (CPIM)",
            "issuer": "Association for Supply Chain Management",
            "url": "https://www.ascm.org/learning-development/certifications-credentials/cpim/",
            "eligibility": "Confirm the current exam version and learning-system options.",
            "why_it_fits": "CPIM covers planning, inventory, demand, supply, quality, continuous improvement, and technology.",
            "practitioner": "James Tilton · CPIM holder",
            "practitioner_insight": "His issuer-published testimonial says CPIM broadened his knowledge and career path.",
            "practitioner_source_type": "Issuer-published holder account",
            "practitioner_url": "https://www.ascm.org/learning-development/certifications-credentials/cpim/",
            "practitioner_caveat": "This is an issuer-selected testimonial and does not isolate the effect of the credential.",
        },
    },
    "fpa_analyst": {
        "title": "Financial Planning & Analysis Analyst",
        "min_experience": 3,
        "preferred_industries": ["financial_services", "retail_ecommerce", "professional_services"],
        "current_demand": demand(
            "Moderate", 0.6, "Sector-to-role inference",
            "Financial services is important in Philippine employment and digitalization, but the reviewed sources do not supply a current national FP&A vacancy count.",
            ["psa_lfs", "dole_forecast"],
        ),
        "future_demand": demand(
            "Mixed", 0.6, "Mixed evidence",
            "AI and analytics can expand scenario work while automating recurring reporting. WEF direction is positive for analytical skills but not an FP&A-specific forecast.",
            ["wef_skills", "ilo_genai"],
        ),
        "ai_opportunity": "Use AI for variance narratives, scenario drafts, and data reconciliation while retaining control of assumptions, accounting meaning, and challenge.",
        "human_edge": "Commercial judgment, governance, scenario framing, and explaining uncertainty to decision-makers.",
        "first_proof": "Create a driver-based forecast with an AI-assisted variance memo, sensitivity analysis, source checks, and explicit assumptions.",
        "certification": {
            "name": "Certified Management Accountant (CMA)",
            "issuer": "Institute of Management Accountants",
            "url": "https://www.imanet.org/ima-certifications/cma-certification",
            "eligibility": "Education, experience, membership, and two exam parts are required.",
            "why_it_fits": "CMA covers planning, performance, analytics, corporate finance, decision analysis, risk, and ethics.",
            "practitioner": "Dylan Kady, CMA · finance practitioner",
            "practitioner_insight": "He reported applying pricing and forecasting learning and later receiving broader opportunities.",
            "practitioner_source_type": "Issuer-published holder account",
            "practitioner_url": "https://www.imanet.org/en/Newsletters/Inside-IMA/2018/April/myCMA-Dylan-Kady",
            "practitioner_caveat": "This is one issuer-published career story; it does not establish causality.",
        },
    },
    "customer_experience_manager": {
        "title": "Customer Experience Manager",
        "min_experience": 4,
        "preferred_industries": ["it_bpo", "retail_ecommerce", "tourism_hospitality"],
        "current_demand": demand(
            "Strong", 0.8, "Sector-to-role inference",
            "DOLE identifies services, IT–BPM/BPO, platform work, and tourism among areas of Philippine demand. CX management is a sector-to-role inference.",
            ["dole_forecast", "psa_lfs"],
        ),
        "future_demand": demand(
            "Mixed", 0.6, "Mixed evidence",
            "Generative AI can automate portions of customer service, but ILO emphasizes task transformation and the continued need for human oversight.",
            ["ilo_genai", "wef_industry"],
        ),
        "ai_opportunity": "Use AI to summarize feedback, assist agents, and identify journey friction while testing accuracy, escalation, consent, and service recovery.",
        "human_edge": "Empathy, cross-functional influence, service recovery, and accountability for customer outcomes.",
        "first_proof": "Redesign one service journey using call/contact evidence, an AI-assisted theme analysis, error checks, and a measured pilot.",
        "certification": {
            "name": "Certified Customer Experience Professional (CCXP)",
            "issuer": "Customer Experience Professionals Association",
            "url": "https://cxpaglobal.org/get-certified",
            "eligibility": "Education and multi-competency CX experience requirements apply.",
            "why_it_fits": "CCXP validates broad experience across customer insight, strategy, design, measurement, and culture.",
            "practitioner": "Mariana De Marchi, CCXP · CX practitioner",
            "practitioner_insight": "She reports finding a role through the CXPA community.",
            "practitioner_source_type": "Issuer-published holder account",
            "practitioner_url": "https://cxpaglobal.org/",
            "practitioner_caveat": "Her account concerns community participation and networking, not proof that the CCXP exam produced the role.",
        },
    },
    "ux_researcher": {
        "title": "UX Researcher",
        "min_experience": 2,
        "preferred_industries": ["it_bpo", "retail_ecommerce", "professional_services"],
        "current_demand": demand(
            "Moderate", 0.6, "Global directional evidence",
            "The reviewed Philippine sources do not provide a UX-researcher vacancy count. This grade reflects digital-product relevance rather than direct national demand evidence.",
            ["dole_forecast", "tesda_5ir"],
        ),
        "future_demand": demand(
            "Mixed", 0.6, "Mixed evidence",
            "WEF's information-technology-services outlook is mixed for design and user-experience roles even as technological literacy and creative thinking rise.",
            ["wef_industry", "wef_skills"],
        ),
        "ai_opportunity": "Use AI for transcript organization, research-ops support, and pattern prompts while protecting consent and validating themes against source evidence.",
        "human_edge": "Study design, rapport, interpretation, accessibility, ethics, and challenging product assumptions.",
        "first_proof": "Run a five-person study with consent, an AI-use disclosure, a traceable evidence table, counterexamples, and design decisions.",
        "certification": {
            "name": "NN/g UX Certification",
            "issuer": "Nielsen Norman Group",
            "url": "https://www.nngroup.com/ux-certification/",
            "eligibility": "Five eligible live courses and corresponding exams are required.",
            "why_it_fits": "The program can combine research, design, and AI-related courses, and it publishes its requirements and cost.",
            "practitioner": "Corey Nunez · UX-certified practitioner",
            "practitioner_insight": "He says the credential added credibility to his decisions in industry.",
            "practitioner_source_type": "Issuer-published holder account",
            "practitioner_url": "https://www.nngroup.com/ux-certification/",
            "practitioner_caveat": "NN/g states that the program is not accredited. Holder comments are selected testimonials and the listed investment is substantial.",
        },
    },
}

VERIFIED_ON = "2026-07-27"
print(
    f"Loaded {len(CAREERS)} careers, {len(EVIDENCE_SOURCES)} research sources, "
    f"and {len(CAREERS)} official credential records."
)


### Evidence grading used by the reranker

- **Direct role evidence:** the source discusses the role or a closely matched occupation.
- **Sector-to-role inference:** the source supports a Philippine sector or technology direction; the notebook explicitly infers the role mapping.
- **Global directional evidence:** useful for 2030 direction, but not a Philippine vacancy count.
- **Mixed evidence:** task automation and augmentation point in different directions.

Scores (`0.6`, `0.8`, `1.0`) are ordinal weights attached to those grades. They are not probabilities and should not be interpreted as labor-demand percentages.


In [ ]:
evidence_overview = pd.DataFrame(
    [
        {
            "career": career["title"],
            "current_grade": career["current_demand"]["label"],
            "current_basis": career["current_demand"]["basis"],
            "future_grade": career["future_demand"]["label"],
            "future_basis": career["future_demand"]["basis"],
        }
        for career in CAREERS.values()
    ]
)
display(evidence_overview)


## 3. Generate 100 synthetic profiles

There are ten profiles per target career. The `PROTOTYPES` below are explicit assumptions—not claims about actual Filipino workers. Each synthetic feature is perturbed around a prototype to create a small, balanced teaching dataset.


In [ ]:
PROTOTYPES = {
    "data_bi_analyst": {
        "industries": ["financial_services", "it_bpo", "retail_ecommerce", "professional_services"],
        "experience": 4.5, "leadership": 0.8,
        "skills": [0.92, 0.78, 0.66, 0.53, 0.34, 0.56, 0.31, 0.52, 0.48],
    },
    "cybersecurity_analyst": {
        "industries": ["it_bpo", "financial_services", "government_public", "professional_services"],
        "experience": 5.0, "leadership": 0.7,
        "skills": [0.60, 0.96, 0.62, 0.55, 0.20, 0.30, 0.20, 0.54, 0.37],
    },
    "cloud_solutions_engineer": {
        "industries": ["it_bpo", "financial_services", "professional_services", "retail_ecommerce"],
        "experience": 5.5, "leadership": 1.0,
        "skills": [0.66, 0.97, 0.66, 0.68, 0.25, 0.28, 0.20, 0.63, 0.42],
    },
    "project_manager": {
        "industries": ["professional_services", "it_bpo", "manufacturing_logistics", "government_public"],
        "experience": 8.0, "leadership": 3.5,
        "skills": [0.58, 0.57, 0.92, 0.97, 0.35, 0.44, 0.55, 0.69, 0.72],
    },
    "digital_marketing_strategist": {
        "industries": ["retail_ecommerce", "it_bpo", "tourism_hospitality", "professional_services"],
        "experience": 4.5, "leadership": 1.0,
        "skills": [0.72, 0.62, 0.88, 0.61, 0.96, 0.35, 0.28, 0.42, 0.82],
    },
    "people_analytics_specialist": {
        "industries": ["it_bpo", "professional_services", "financial_services", "healthcare"],
        "experience": 5.5, "leadership": 1.2,
        "skills": [0.84, 0.58, 0.86, 0.58, 0.32, 0.35, 0.97, 0.45, 0.64],
    },
    "supply_chain_analyst": {
        "industries": ["manufacturing_logistics", "retail_ecommerce", "professional_services", "other"],
        "experience": 5.0, "leadership": 1.0,
        "skills": [0.84, 0.57, 0.68, 0.66, 0.27, 0.51, 0.26, 0.97, 0.56],
    },
    "fpa_analyst": {
        "industries": ["financial_services", "professional_services", "retail_ecommerce", "manufacturing_logistics"],
        "experience": 5.5, "leadership": 1.0,
        "skills": [0.87, 0.54, 0.67, 0.63, 0.25, 0.98, 0.28, 0.56, 0.43],
    },
    "customer_experience_manager": {
        "industries": ["it_bpo", "retail_ecommerce", "tourism_hospitality", "financial_services"],
        "experience": 7.0, "leadership": 3.0,
        "skills": [0.52, 0.51, 0.95, 0.78, 0.50, 0.37, 0.55, 0.65, 0.98],
    },
    "ux_researcher": {
        "industries": ["it_bpo", "retail_ecommerce", "professional_services", "education"],
        "experience": 4.5, "leadership": 0.8,
        "skills": [0.75, 0.65, 0.94, 0.57, 0.93, 0.25, 0.45, 0.38, 0.88],
    },
}

def clip_rating(value):
    return int(np.clip(np.rint(value), 1, 5))

def generate_synthetic_profiles(seed=SEED, per_career=10):
    rng = np.random.default_rng(seed)
    rows = []
    profile_number = 1

    for career_id, prototype in PROTOTYPES.items():
        for _ in range(per_career):
            if rng.random() < 0.82:
                industry = rng.choice(prototype["industries"])
            else:
                industry = rng.choice(INDUSTRIES)

            years = float(np.clip(rng.normal(prototype["experience"], 1.8), 0, 20))
            leadership = float(
                np.clip(
                    rng.normal(prototype["leadership"], 0.9),
                    0,
                    min(12, years),
                )
            )

            row = {
                "profile_id": f"PH-{profile_number:03d}",
                "current_industry": industry,
                "years_experience": round(years, 1),
                "leadership_years": round(leadership, 1),
            }
            for skill, center in zip(SKILL_FEATURES, prototype["skills"]):
                row[skill] = clip_rating(1 + 4 * center + rng.normal(0, 0.48))

            row["recommended_career"] = career_id
            rows.append(row)
            profile_number += 1

    return pd.DataFrame(rows)

profiles = generate_synthetic_profiles()
assert len(profiles) == 100
assert profiles["profile_id"].nunique() == 100
assert profiles["recommended_career"].value_counts().eq(10).all()
assert profiles[SKILL_FEATURES].apply(lambda col: col.between(1, 5).all()).all()

profiles.to_csv("hakbang_ph_100_synthetic_profiles.csv", index=False)
display(profiles.head())
display(
    profiles["recommended_career"]
    .map({key: value["title"] for key, value in CAREERS.items()})
    .value_counts()
    .rename_axis("career")
    .to_frame("profiles")
)
print("Saved: hakbang_ph_100_synthetic_profiles.csv")


## 4. Compare candidate models

Because each class has ten records, stratified 5-fold cross-validation places eight records per class in training and two in validation on each fold. We rank models by **macro-F1** (equal importance for every career), with accuracy as the tie-breaker.


In [ ]:
def compatible_one_hot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:  # scikit-learn < 1.2
        return OneHotEncoder(handle_unknown="ignore", sparse=False)

def make_preprocessor():
    return ColumnTransformer(
        transformers=[
            ("numeric", StandardScaler(), NUMERIC_FEATURES),
            ("industry", compatible_one_hot_encoder(), CATEGORICAL_FEATURES),
        ],
        remainder="drop",
    )

candidate_models = {
    "Gaussian Naive Bayes": GaussianNB(),
    "Logistic Regression": LogisticRegression(
        max_iter=3000,
        class_weight="balanced",
        random_state=SEED,
    ),
    "7-NN (distance weighted)": KNeighborsClassifier(
        n_neighbors=7,
        weights="distance",
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=400,
        class_weight="balanced_subsample",
        random_state=SEED,
        n_jobs=-1,
    ),
    "Extra Trees": ExtraTreesClassifier(
        n_estimators=400,
        class_weight="balanced",
        random_state=SEED,
        n_jobs=-1,
    ),
}

X = profiles[CATEGORICAL_FEATURES + NUMERIC_FEATURES]
y = profiles["recommended_career"]
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

benchmark_rows = []
for model_name, estimator in candidate_models.items():
    pipeline = Pipeline(
        [
            ("preprocess", make_preprocessor()),
            ("model", estimator),
        ]
    )
    scores = cross_validate(
        pipeline,
        X,
        y,
        cv=cv,
        scoring={"accuracy": "accuracy", "macro_f1": "f1_macro"},
        n_jobs=1,
        error_score="raise",
    )
    benchmark_rows.append(
        {
            "model": model_name,
            "mean_macro_f1": scores["test_macro_f1"].mean(),
            "std_macro_f1": scores["test_macro_f1"].std(),
            "mean_accuracy": scores["test_accuracy"].mean(),
            "std_accuracy": scores["test_accuracy"].std(),
        }
    )

benchmark = (
    pd.DataFrame(benchmark_rows)
    .sort_values(["mean_macro_f1", "mean_accuracy"], ascending=False)
    .reset_index(drop=True)
)
benchmark.to_csv("hakbang_ph_model_benchmark.csv", index=False)

display(
    benchmark.assign(
        mean_macro_f1=benchmark["mean_macro_f1"].round(3),
        std_macro_f1=benchmark["std_macro_f1"].round(3),
        mean_accuracy=benchmark["mean_accuracy"].round(3),
        std_accuracy=benchmark["std_accuracy"].round(3),
    )
)

ax = benchmark.sort_values("mean_macro_f1").plot.barh(
    x="model",
    y=["mean_macro_f1", "mean_accuracy"],
    figsize=(9, 5),
    xlim=(0, 1),
    color=["#0b6e69", "#e5983e"],
)
ax.set_title("Synthetic 5-fold cross-validation")
ax.set_xlabel("Score")
ax.set_ylabel("")
plt.legend(["Macro-F1", "Accuracy"], loc="lower right")
plt.tight_layout()
plt.show()


## 5. Inspect the selected model and train on all 100 profiles

The following report is based on out-of-fold predictions. It is useful for checking internal consistency on the synthetic sample, but it is not an estimate of real-world effectiveness.


In [ ]:
best_model_name = benchmark.loc[0, "model"]
best_pipeline = Pipeline(
    [
        ("preprocess", make_preprocessor()),
        ("model", deepcopy(candidate_models[best_model_name])),
    ]
)

out_of_fold_predictions = cross_val_predict(
    best_pipeline,
    X,
    y,
    cv=cv,
    n_jobs=1,
)

print("Selected model:", best_model_name)
print(
    classification_report(
        y,
        out_of_fold_predictions,
        target_names=[CAREERS[class_id]["title"] for class_id in sorted(CAREERS)],
        zero_division=0,
    )
)

label_order = sorted(CAREERS)
matrix = confusion_matrix(y, out_of_fold_predictions, labels=label_order)
plt.figure(figsize=(11, 8))
sns.heatmap(
    matrix,
    annot=True,
    fmt="d",
    cmap="BuGn",
    xticklabels=[CAREERS[key]["title"] for key in label_order],
    yticklabels=[CAREERS[key]["title"] for key in label_order],
)
plt.title("Out-of-fold confusion matrix — synthetic data only")
plt.xlabel("Predicted")
plt.ylabel("Synthetic label")
plt.xticks(rotation=50, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

best_pipeline.fit(X, y)
print(f"Fitted {best_model_name} on all {len(profiles)} synthetic profiles.")


## 6. Recommendation function

The first stage predicts relative fit from experience, current industry, and skills. The second stage is deliberately transparent:

- synthetic model fit: **38%**
- direct skill fit: **24%**
- experience fit: **8%**
- current-demand evidence grade: **8%**
- future-demand evidence grade: **12%**
- industry fit: **6%**
- career-goal adjustment: up to **4%**

The **recommendation score is a comparative index**, not the probability of being hired or succeeding. Skill gaps compare the user's 1–5 rating with this notebook's explicit prototype target.


In [ ]:
CAREER_GOALS = [
    "future_ready",
    "leadership",
    "sector_switch",
    "build_specialty",
]

def validate_profile(profile):
    missing_fields = [
        field
        for field in CATEGORICAL_FEATURES + NUMERIC_FEATURES + ["goal"]
        if field not in profile
    ]
    if missing_fields:
        raise ValueError(f"Missing profile fields: {missing_fields}")
    if profile["current_industry"] not in INDUSTRIES:
        raise ValueError(
            "current_industry must be one of: " + ", ".join(INDUSTRIES)
        )
    for skill in SKILL_FEATURES:
        if not 1 <= float(profile[skill]) <= 5:
            raise ValueError(f"{skill} must be between 1 and 5")
    if float(profile["years_experience"]) < 0:
        raise ValueError("years_experience cannot be negative")
    if not 0 <= float(profile["leadership_years"]) <= float(profile["years_experience"]):
        raise ValueError(
            "leadership_years must be between 0 and years_experience"
        )
    if profile["goal"] not in CAREER_GOALS:
        raise ValueError("goal must be one of: " + ", ".join(CAREER_GOALS))

def target_ratings(career_id):
    return {
        skill: int(np.clip(np.rint(1 + 4 * center), 1, 5))
        for skill, center in zip(
            SKILL_FEATURES,
            PROTOTYPES[career_id]["skills"],
        )
    }

def skill_gaps_for(profile, career_id, limit=3):
    targets = target_ratings(career_id)
    gaps = [
        (skill, target, float(profile[skill]), target - float(profile[skill]))
        for skill, target in targets.items()
        if target - float(profile[skill]) > 0
    ]
    gaps.sort(key=lambda item: item[3], reverse=True)
    return [
        f"{skill.replace('_', ' ').title()} (current {current:g}/5; target {target}/5)"
        for skill, target, current, _ in gaps[:limit]
    ]

def evidence_links(source_keys):
    return " · ".join(
        f'[{EVIDENCE_SOURCES[key]["name"]}]({EVIDENCE_SOURCES[key]["url"]})'
        for key in source_keys
    )

def recommend_careers(profile, top_k=3):
    validate_profile(profile)
    input_frame = pd.DataFrame([profile])[
        CATEGORICAL_FEATURES + NUMERIC_FEATURES
    ]
    probabilities = best_pipeline.predict_proba(input_frame)[0]
    class_ids = best_pipeline.named_steps["model"].classes_

    rows = []
    for career_id, model_probability in zip(class_ids, probabilities):
        career = CAREERS[career_id]
        current_evidence = career["current_demand"]
        future_evidence = career["future_demand"]
        targets = target_ratings(career_id)
        skill_fit = 1 - np.mean(
            [
                abs(float(profile[skill]) - target) / 4
                for skill, target in targets.items()
            ]
        )
        experience_fit = min(
            1.0,
            float(profile["years_experience"])
            / max(career["min_experience"], 1),
        )
        industry_fit = (
            1.0
            if profile["current_industry"] in career["preferred_industries"]
            else 0.55
        )
        goal_adjustment = 0.0
        if (
            profile["goal"] == "leadership"
            and career_id in {"project_manager", "customer_experience_manager"}
        ):
            goal_adjustment = 0.04
        elif profile["goal"] == "future_ready":
            goal_adjustment = max(
                0.0,
                future_evidence["score"] - current_evidence["score"],
            ) * 0.04
        elif profile["goal"] == "sector_switch" and industry_fit < 1:
            goal_adjustment = future_evidence["score"] * 0.04

        recommendation_score = 100 * min(
            1.0,
            0.38 * model_probability
            + 0.24 * skill_fit
            + 0.08 * experience_fit
            + 0.08 * current_evidence["score"]
            + 0.12 * future_evidence["score"]
            + 0.06 * industry_fit
            + goal_adjustment,
        )
        certification = career["certification"]
        gaps = skill_gaps_for(profile, career_id)

        rows.append(
            {
                "career_id": career_id,
                "career": career["title"],
                "recommendation_score": round(recommendation_score, 1),
                "synthetic_model_confidence": round(100 * model_probability, 1),
                "skill_fit_index": round(100 * skill_fit, 1),
                "experience_fit_index": round(100 * experience_fit, 1),
                "industry_fit_index": round(100 * industry_fit, 1),
                "current_demand_grade": current_evidence["label"],
                "current_evidence_basis": current_evidence["basis"],
                "current_evidence_insight": current_evidence["insight"],
                "current_evidence_sources": evidence_links(current_evidence["sources"]),
                "future_demand_grade": future_evidence["label"],
                "future_evidence_basis": future_evidence["basis"],
                "future_evidence_insight": future_evidence["insight"],
                "future_evidence_sources": evidence_links(future_evidence["sources"]),
                "ai_opportunity": career["ai_opportunity"],
                "human_edge": career["human_edge"],
                "first_proof": career["first_proof"],
                "priority_skill_gaps": "; ".join(gaps) if gaps else "No gap against demo target",
                "certification": certification["name"],
                "issuer": certification["issuer"],
                "official_url": certification["url"],
                "certification_eligibility": certification["eligibility"],
                "why_certification_fits": certification["why_it_fits"],
                "practitioner": certification["practitioner"],
                "practitioner_insight": certification["practitioner_insight"],
                "practitioner_source_type": certification["practitioner_source_type"],
                "practitioner_url": certification["practitioner_url"],
                "practitioner_caveat": certification["practitioner_caveat"],
                "link_last_checked": VERIFIED_ON,
            }
        )

    return (
        pd.DataFrame(rows)
        .sort_values(
            ["recommendation_score", "synthetic_model_confidence"],
            ascending=False,
        )
        .head(top_k)
        .reset_index(drop=True)
    )

demo_profile = {
    "current_industry": "it_bpo",
    "years_experience": 5.0,
    "leadership_years": 1.0,
    "goal": "future_ready",
    "data_analytics": 4,
    "technology": 4,
    "communication": 4,
    "project_management": 3,
    "creative_design": 2,
    "finance": 2,
    "people_hr": 2,
    "operations": 3,
    "customer_experience": 3,
}
display(recommend_careers(demo_profile))


## 7. Enter a professional profile

In Colab, change the form fields and run this cell. Skill ratings use `1 = beginner` through `5 = advanced`.


In [ ]:
current_industry = "it_bpo" #@param ["it_bpo", "financial_services", "manufacturing_logistics", "retail_ecommerce", "government_public", "healthcare", "education", "tourism_hospitality", "professional_services", "other"]
years_experience = 6.0 #@param {type:"number", min:0, max:40, step:0.5}
leadership_years = 1.5 #@param {type:"number", min:0, max:30, step:0.5}
goal = "future_ready" #@param ["future_ready", "leadership", "sector_switch", "build_specialty"]
data_analytics = 4 #@param {type:"slider", min:1, max:5, step:1}
technology = 4 #@param {type:"slider", min:1, max:5, step:1}
communication = 4 #@param {type:"slider", min:1, max:5, step:1}
project_management = 3 #@param {type:"slider", min:1, max:5, step:1}
creative_design = 2 #@param {type:"slider", min:1, max:5, step:1}
finance = 2 #@param {type:"slider", min:1, max:5, step:1}
people_hr = 2 #@param {type:"slider", min:1, max:5, step:1}
operations = 3 #@param {type:"slider", min:1, max:5, step:1}
customer_experience = 4 #@param {type:"slider", min:1, max:5, step:1}

user_profile = {
    "current_industry": current_industry,
    "years_experience": years_experience,
    "leadership_years": leadership_years,
    "goal": goal,
    "data_analytics": data_analytics,
    "technology": technology,
    "communication": communication,
    "project_management": project_management,
    "creative_design": creative_design,
    "finance": finance,
    "people_hr": people_hr,
    "operations": operations,
    "customer_experience": customer_experience,
}

recommendations = recommend_careers(user_profile, top_k=3)
display(
    recommendations[
        [
            "career",
            "recommendation_score",
            "synthetic_model_confidence",
            "skill_fit_index",
            "current_demand_grade",
            "future_demand_grade",
            "priority_skill_gaps",
        ]
    ]
)

for rank, row in recommendations.iterrows():
    display(
        Markdown(
            f'### {rank + 1}. {row["career"]}\n\n'
            f'**Why it is relevant now — {row["current_demand_grade"]} '
            f'({row["current_evidence_basis"]}):** '
            f'{row["current_evidence_insight"]}  \n'
            f'**Current-demand sources:** {row["current_evidence_sources"]}  \n\n'
            f'**Future direction — {row["future_demand_grade"]} '
            f'({row["future_evidence_basis"]}):** '
            f'{row["future_evidence_insight"]}  \n'
            f'**Future-demand sources:** {row["future_evidence_sources"]}  \n\n'
            f'**AI-era opportunity:** {row["ai_opportunity"]}  \n'
            f'**Human edge:** {row["human_edge"]}  \n'
            f'**Portfolio proof:** {row["first_proof"]}  \n'
            f'**Priority skill gaps:** {row["priority_skill_gaps"]}  \n\n'
            f'**Certification:** [{row["certification"]}]({row["official_url"]}) '
            f'— {row["issuer"]}  \n'
            f'**Why this credential fits:** {row["why_certification_fits"]}  \n'
            f'**Eligibility note:** {row["certification_eligibility"]}  \n'
            f'**Practitioner evidence — {row["practitioner_source_type"]}:** '
            f'[{row["practitioner"]}]({row["practitioner_url"]}) — '
            f'{row["practitioner_insight"]}  \n'
            f'**Evidence caveat:** {row["practitioner_caveat"]}  \n'
            f'**Credential link last checked:** {row["link_last_checked"]}'
        )
    )


## 8. Validate and export artifacts

The validation cell checks data shape, required fields, HTTPS certification URLs, model class coverage, and the recommendation output. It then saves a portable model bundle. The `download_artifacts()` helper works in Colab but does not start downloads automatically.


In [ ]:
assert profiles.shape[0] == 100
assert not profiles[CATEGORICAL_FEATURES + NUMERIC_FEATURES + ["recommended_career"]].isna().any().any()
assert set(best_pipeline.named_steps["model"].classes_) == set(CAREERS)
assert set(PROTOTYPES) == set(CAREERS)
assert len(EVIDENCE_SOURCES) == 9
assert VERIFIED_ON == "2026-07-27"

for career_id, career in CAREERS.items():
    certificate = career["certification"]
    parsed = urlparse(certificate["url"])
    assert parsed.scheme == "https" and parsed.netloc
    assert certificate["name"] and certificate["issuer"]
    assert certificate["practitioner_source_type"] in {
        "Issuer-published holder account",
        "Issuer survey",
        "Public first-person account",
    }
    assert certificate["practitioner_caveat"]
    assert urlparse(certificate["practitioner_url"]).scheme == "https"
    for demand_key in ["current_demand", "future_demand"]:
        evidence = career[demand_key]
        assert evidence["basis"] in {
            "Direct role evidence",
            "Sector-to-role inference",
            "Global directional evidence",
            "Mixed evidence",
        }
        assert evidence["score"] in {0.6, 0.8, 1.0}
        assert evidence["sources"]
        assert all(key in EVIDENCE_SOURCES for key in evidence["sources"])

for source in EVIDENCE_SOURCES.values():
    parsed = urlparse(source["url"])
    assert parsed.scheme == "https" and parsed.netloc

smoke_test = recommend_careers(demo_profile, top_k=3)
assert len(smoke_test) == 3
assert smoke_test["recommendation_score"].between(0, 100).all()
assert smoke_test["official_url"].str.startswith("https://").all()
assert smoke_test["practitioner_url"].str.startswith("https://").all()
assert smoke_test["current_evidence_sources"].str.contains("https://").all()
assert smoke_test["future_evidence_sources"].str.contains("https://").all()

model_bundle = {
    "pipeline": best_pipeline,
    "features": CATEGORICAL_FEATURES + NUMERIC_FEATURES,
    "careers": CAREERS,
    "evidence_sources": EVIDENCE_SOURCES,
    "prototypes": PROTOTYPES,
    "verified_on": VERIFIED_ON,
    "training_data_notice": "Trained only on 100 synthetic profiles.",
    "selected_model": best_model_name,
    "seed": SEED,
    "dataset_version": DATASET_VERSION,
}
joblib.dump(model_bundle, "hakbang_ph_career_recommender.joblib")

print("All validation checks passed.")
print("Saved:")
print("- hakbang_ph_100_synthetic_profiles.csv")
print("- hakbang_ph_model_benchmark.csv")
print("- hakbang_ph_career_recommender.joblib")

def download_artifacts():
    try:
        from google.colab import files
    except ImportError:
        print("This helper downloads files only when the notebook runs in Google Colab.")
        return
    for filename in [
        "hakbang_ph_100_synthetic_profiles.csv",
        "hakbang_ph_model_benchmark.csv",
        "hakbang_ph_career_recommender.joblib",
    ]:
        files.download(filename)


## 9. Publish on GitHub and open in Colab

This notebook is self-contained, so it can be committed directly to a GitHub repository.

1. Commit `notebooks/Hakbang_PH_Career_Recommender_Colab.ipynb`.
2. Push the repository to GitHub.
3. Open it through this URL pattern, replacing `OWNER` and `REPOSITORY`:

`https://colab.research.google.com/github/OWNER/REPOSITORY/blob/main/notebooks/Hakbang_PH_Career_Recommender_Colab.ipynb`

GitHub can render the notebook without executing it. Colab users should run the cells from top to bottom so the dataset, benchmark, fitted model, and recommendations are regenerated from the documented seed.


## Before production use

Replace the synthetic target labels and ordinal evidence grades with governed production data:

1. collect consented, de-identified career-transition outcomes and define what a successful move means;
2. connect licensed, time-stamped Philippine vacancy and occupational data, with region and seniority;
3. use temporal holdouts and an untouched external test set—not only random cross-validation;
4. calibrate model probabilities and audit performance by legally and ethically appropriate groups;
5. review skill-gap explanations with Philippine career practitioners and industry experts;
6. revalidate every certification against its issuer before showing it;
7. include human review, user correction, appeal, privacy, retention, and security controls.

No model can honestly promise the “best” career move from 100 synthetic records. This notebook is a reproducible starting point for collecting better evidence and testing the end-to-end workflow.
